In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from tqdm import tqdm
import logging

# For modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

# Set up logging for errors during descriptor computation
logging.basicConfig(filename='descriptor_errors.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

# =============================================================================
# 1. Load Data and Prepare Molecules
# =============================================================================
data_df = pd.read_csv("./dw_data/Opt1_acidic_tr.csv")
smiles_col = 'OriginalSmiles'
target = 'pKa'

# Convert SMILES to RDKit Mol objects and remove salts
saltRemover = SaltRemover(defnFilename='./Salts.txt')
mols = data_df[smiles_col].astype(str).apply(Chem.MolFromSmiles).apply(saltRemover.StripMol)

# =============================================================================
# 2. Compute a Larger Set of Selected Descriptors
# =============================================================================
# Define a list of selected descriptor names (more than the previous 11)
selected_descriptor_names = [
    "MolWt", "MolLogP", "NumHDonors", "NumHAcceptors", "TPSA", 
    "NumRotatableBonds", "LabuteASA", "BertzCT", "Ipc", "MolMR", 
    "BalabanJ", "Chi0", "Chi1", "Chi2n", "Chi3n", "Kappa1", "Kappa2", "Kappa3"
]

# Create a dictionary mapping descriptor names to functions from RDKit
selected_desc_funcs = {name: func for name, func in Descriptors.descList if name in selected_descriptor_names}

desc_data = []
print("Computing selected descriptors for each molecule...")
for i, mol in tqdm(enumerate(mols), total=len(mols)):
    desc_values = {}
    for name, func in selected_desc_funcs.items():
        try:
            value = func(mol) if mol is not None else np.nan
            desc_values[name] = value
        except Exception as e:
            logging.info(f"Molecule {i}, descriptor {name}: {e}")
            desc_values[name] = np.nan
    desc_data.append(desc_values)

desc_df = pd.DataFrame(desc_data)

# =============================================================================
# 3. Clean Descriptor Data and Save to Excel
# =============================================================================
# Drop molecules (rows) with any missing descriptor values.
desc_df_clean = desc_df.dropna().reset_index(drop=True)
# Remove descriptor columns with zero variance.
desc_df_clean = desc_df_clean.loc[:, desc_df_clean.std() > 0]

# Subset target values using the cleaned indices.
y_desc = data_df.loc[desc_df_clean.index, target].values

# Save the computed descriptors to an Excel file.
desc_df_clean.to_excel("selected_descriptors.xlsx", index=False)
print("Selected descriptors saved to selected_descriptors.xlsx")
print(f"Final descriptor matrix shape: {desc_df_clean.shape}")

# =============================================================================
# 4. Evaluate Each Descriptor Individually Using XGBoost
# =============================================================================
# Hyperparameter grid for XGBoost
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

def train_and_evaluate_single_descriptor(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=11)
    xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
    grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                               scoring='r2', cv=5, n_jobs=5, verbose=0)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_rmse = np.sqrt(test_mse)
    return {
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_MSE': test_mse,
        'Test_RMSE': test_rmse,
        'Best_Params': str(grid_search.best_params_)
    }

performance_list = []
print("Evaluating each descriptor individually...")
for col in tqdm(desc_df_clean.columns, total=len(desc_df_clean.columns)):
    try:
        X_single = desc_df_clean[[col]].values.astype(float)
        metrics = train_and_evaluate_single_descriptor(X_single, y_desc)
        metrics['Descriptor'] = col
        performance_list.append(metrics)
    except Exception as e:
        logging.info(f"Error evaluating descriptor {col}: {e}")

performance_df = pd.DataFrame(performance_list)
performance_df.sort_values(by='Test_R2', ascending=False, inplace=True)

# Save individual descriptor performance to an Excel file.
performance_df.to_excel("individual_descriptor_performance.xlsx", index=False)
print("Individual descriptor performance saved to individual_descriptor_performance.xlsx")

print("\nBest individual descriptors based on Test R2:")
print(performance_df[['Descriptor', 'Test_R2']].head())

# =============================================================================
# 5. Combine the Best Descriptors and Train a Combined Model
# =============================================================================
# For example, select the top 5 descriptors based on Test R2
num_best = 5
best_descriptors = performance_df.head(num_best)['Descriptor'].tolist()
print(f"\nCombining the top {num_best} descriptors: {best_descriptors}")

# Create a combined feature matrix using the best descriptors
X_combined = desc_df_clean[best_descriptors].values.astype(float)

# Train/test split and model training for combined features
X_train, X_test, y_train, y_test = train_test_split(X_combined, y_desc, test_size=0.25, random_state=11)
xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                           scoring='r2', cv=5, n_jobs=5, verbose=0)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

combined_metrics = {
    'Train_R2': r2_score(y_train, y_pred_train),
    'Test_R2': r2_score(y_test, y_pred_test),
    'Test_MAE': mean_absolute_error(y_test, y_pred_test),
    'Test_MSE': mean_squared_error(y_test, y_pred_test),
    'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    'Best_Params': str(grid_search.best_params_)
}

print("\nCombined model performance using top descriptors:")
for k, v in combined_metrics.items():
    print(f"{k}: {v}")

# Optionally, save the combined model performance to an Excel file
combined_df = pd.DataFrame([combined_metrics])
combined_df.to_excel("combined_descriptor_model_performance.xlsx", index=False)
print("Combined model performance saved to combined_descriptor_model_performance.xlsx")


Computing selected descriptors for each molecule...


100%|████████████████████████████████████████████████████████████████████████████| 2220/2220 [00:01<00:00, 1197.06it/s]


Selected descriptors saved to selected_descriptors.xlsx
Final descriptor matrix shape: (2219, 18)
Evaluating each descriptor individually...


100%|██████████████████████████████████████████████████████████████████████████████████| 18/18 [00:29<00:00,  1.62s/it]


Individual descriptor performance saved to individual_descriptor_performance.xlsx

Best individual descriptors based on Test R2:
   Descriptor   Test_R2
12       TPSA  0.410128
4        Chi1  0.080118
0       MolWt  0.058962
7         Ipc  0.055891
17      MolMR  0.047700

Combining the top 5 descriptors: ['TPSA', 'Chi1', 'MolWt', 'Ipc', 'MolMR']

Combined model performance using top descriptors:
Train_R2: 0.9672771200168174
Test_R2: 0.4815021546578334
Test_MAE: 1.725963995549056
Test_MSE: 5.881127612172139
Test_RMSE: 2.4251036291614714
Best_Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}
Combined model performance saved to combined_descriptor_model_performance.xlsx


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"


In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from tqdm import tqdm
import logging

# For modeling
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

# Set up logging for errors during descriptor computation
logging.basicConfig(filename='descriptor_errors.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

# =============================================================================
# 1. Load Data and Prepare Molecules
# =============================================================================
data_df = pd.read_csv("./dw_data/Opt1_acidic_tr.csv")
smiles_col = 'OriginalSmiles'
target = 'pKa'

# Convert SMILES to RDKit Mol objects and remove salts
saltRemover = SaltRemover(defnFilename='./Salts.txt')
mols = data_df[smiles_col].astype(str).apply(Chem.MolFromSmiles).apply(saltRemover.StripMol)

# =============================================================================
# 2. Compute a Larger Set of Selected Descriptors
# =============================================================================
# Define a list of selected descriptor names (you can adjust or expand this list)
selected_descriptor_names = [
    "MolWt", "MolLogP", "NumHDonors", "NumHAcceptors", "TPSA", 
    "NumRotatableBonds", "LabuteASA", "BertzCT", "Ipc", "MolMR", 
    "BalabanJ", "Chi0", "Chi1", "Chi2n", "Chi3n", "Kappa1", "Kappa2", "Kappa3"
]

# Create a dictionary mapping descriptor names to functions from RDKit
selected_desc_funcs = {name: func for name, func in Descriptors.descList if name in selected_descriptor_names}

desc_data = []
print("Computing selected descriptors for each molecule...")
for i, mol in tqdm(enumerate(mols), total=len(mols)):
    desc_values = {}
    for name, func in selected_desc_funcs.items():
        try:
            value = func(mol) if mol is not None else np.nan
            desc_values[name] = value
        except Exception as e:
            logging.info(f"Molecule {i}, descriptor {name}: {e}")
            desc_values[name] = np.nan
    desc_data.append(desc_values)

desc_df = pd.DataFrame(desc_data)

# =============================================================================
# 3. Clean Descriptor Data and Save to Excel
# =============================================================================
# Drop molecules (rows) with any missing descriptor values.
desc_df_clean = desc_df.dropna().reset_index(drop=True)
# Remove descriptor columns with zero variance.
desc_df_clean = desc_df_clean.loc[:, desc_df_clean.std() > 0]

# Subset target values using the cleaned indices.
y_desc = data_df.loc[desc_df_clean.index, target].values

# Save the computed descriptors to an Excel file.
desc_df_clean.to_excel("selected_descriptors.xlsx", index=False)
print("Selected descriptors saved to selected_descriptors.xlsx")
print(f"Final descriptor matrix shape: {desc_df_clean.shape}")

# =============================================================================
# 4. Evaluate Each Descriptor Individually Using XGBoost
# =============================================================================
# Hyperparameter grid for XGBoost
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

def train_and_evaluate_single_descriptor(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=11)
    xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
    grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                               scoring='r2', cv=5, n_jobs=5, verbose=0)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_rmse = np.sqrt(test_mse)
    return {
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_MSE': test_mse,
        'Test_RMSE': test_rmse,
        'Best_Params': str(grid_search.best_params_)
    }

performance_list = []
print("Evaluating each descriptor individually...")
for col in tqdm(desc_df_clean.columns, total=len(desc_df_clean.columns)):
    try:
        X_single = desc_df_clean[[col]].values.astype(float)
        metrics = train_and_evaluate_single_descriptor(X_single, y_desc)
        metrics['Descriptor'] = col
        performance_list.append(metrics)
    except Exception as e:
        logging.info(f"Error evaluating descriptor {col}: {e}")

performance_df = pd.DataFrame(performance_list)
performance_df.sort_values(by='Test_R2', ascending=False, inplace=True)

# Save individual descriptor performance to an Excel file.
performance_df.to_excel("individual_descriptor_performance.xlsx", index=False)
print("Individual descriptor performance saved to individual_descriptor_performance.xlsx")

print("\nBest individual descriptors based on Test R2:")
print(performance_df[['Descriptor', 'Test_R2']].head())

# =============================================================================
# 5. Greedy Forward Selection Among Top 25 Descriptors
# =============================================================================
# We use the top 25 descriptors (from individual performance ranking) as candidates.
top_25 = performance_df.head(25)['Descriptor'].tolist()
print("\nTop 25 descriptors to consider:")
print(top_25)

# Use only the top 25 columns from the cleaned descriptor dataframe.
X_top25 = desc_df_clean[top_25]

def evaluate_subset(feature_subset, X, y):
    """Return mean 5-fold cross-validated R2 score for the given feature subset."""
    model = xgb.XGBRegressor(random_state=42, objective='reg:squarederror', n_estimators=300)
    # 5-fold CV using r2 scoring.
    scores = cross_val_score(model, X[feature_subset], y, cv=5, scoring='r2', n_jobs=1)
    return np.mean(scores)

selected_features = []
current_score = -np.inf
remaining_features = top_25.copy()

print("\nStarting forward selection among top 25 descriptors...")
while remaining_features:
    best_candidate = None
    best_candidate_score = current_score
    for candidate in remaining_features:
        trial_features = selected_features + [candidate]
        score = evaluate_subset(trial_features, X_top25, y_desc)
        # Uncomment below to print each candidate's performance:
        # print(f"Trial {trial_features}: CV R2 = {score:.3f}")
        if score > best_candidate_score:
            best_candidate_score = score
            best_candidate = candidate
    if best_candidate is not None and best_candidate_score > current_score:
        selected_features.append(best_candidate)
        remaining_features.remove(best_candidate)
        current_score = best_candidate_score
        print(f"Added {best_candidate}, subset: {selected_features}, CV R2: {current_score:.3f}")
    else:
        break

print("\nFinal selected descriptor subset:")
print(selected_features)
print(f"Best cross-validated R2 from forward selection: {current_score:.3f}")

# =============================================================================
# 6. Train Final Combined Model Using the Selected Subset
# =============================================================================
X_final = X_top25[selected_features].values.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X_final, y_desc, test_size=0.25, random_state=11)
xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                           scoring='r2', cv=5, n_jobs=5, verbose=0)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)
final_metrics = {
    'Train_R2': r2_score(y_train, y_pred_train),
    'Test_R2': r2_score(y_test, y_pred_test),
    'Test_MAE': mean_absolute_error(y_test, y_pred_test),
    'Test_MSE': mean_squared_error(y_test, y_pred_test),
    'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    'Best_Params': str(grid_search.best_params_)
}

print("\nFinal model performance on the selected subset:")
for k, v in final_metrics.items():
    print(f"{k}: {v}")

# Optionally, save final results.
results_df = pd.DataFrame([{
    'Selected_Descriptors': ', '.join(selected_features),
    **final_metrics
}])
results_df.to_excel("final_selected_descriptor_model_performance.xlsx", index=False)
print("Final model performance saved to final_selected_descriptor_model_performance.xlsx")


Computing selected descriptors for each molecule...


100%|████████████████████████████████████████████████████████████████████████████| 2220/2220 [00:01<00:00, 1197.43it/s]


Selected descriptors saved to selected_descriptors.xlsx
Final descriptor matrix shape: (2219, 18)
Evaluating each descriptor individually...


100%|██████████████████████████████████████████████████████████████████████████████████| 18/18 [00:28<00:00,  1.56s/it]


Individual descriptor performance saved to individual_descriptor_performance.xlsx

Best individual descriptors based on Test R2:
   Descriptor   Test_R2
12       TPSA  0.410128
4        Chi1  0.080118
0       MolWt  0.058962
7         Ipc  0.055891
17      MolMR  0.047700

Top 25 descriptors to consider:
['TPSA', 'Chi1', 'MolWt', 'Ipc', 'MolMR', 'MolLogP', 'LabuteASA', 'BalabanJ', 'Chi3n', 'BertzCT', 'NumHDonors', 'Chi0', 'Chi2n', 'NumRotatableBonds', 'NumHAcceptors', 'Kappa1', 'Kappa3', 'Kappa2']

Starting forward selection among top 25 descriptors...
Added TPSA, subset: ['TPSA'], CV R2: -16.957
Added NumHAcceptors, subset: ['TPSA', 'NumHAcceptors'], CV R2: -16.034
Added NumRotatableBonds, subset: ['TPSA', 'NumHAcceptors', 'NumRotatableBonds'], CV R2: -15.447
Added NumHDonors, subset: ['TPSA', 'NumHAcceptors', 'NumRotatableBonds', 'NumHDonors'], CV R2: -15.275
Added BalabanJ, subset: ['TPSA', 'NumHAcceptors', 'NumRotatableBonds', 'NumHDonors', 'BalabanJ'], CV R2: -14.900
Added LabuteA